# Auto-Labeler Training on Colab (A100)

1D U-Net for offline tissue segmentation — trains on 85 labeled studies.

**Setup:** Data is on Google Drive at `G:\My Drive\auto_labeler` (mounted as `/content/drive/MyDrive/auto_labeler`).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!rm -rf /content/drive/MyDrive/auto_labeler/__pycache__
!cd /tmp && rm -rf PyTorch_3 && git clone https://github.com/RonInQu/PyTorch_3.git
!cp /tmp/PyTorch_3/auto_labeler/*.py /content/drive/MyDrive/auto_labeler/
print("Code synced from git")

Cloning into 'PyTorch_3'...
remote: Enumerating objects: 767, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 767 (delta 116), reused 164 (delta 94), pack-reused 563 (from 1)
Receiving objects: 100% (767/767), 336.24 KiB | 25.86 MiB/s, done.
Resolving deltas: 100% (470/470), done.
Code synced from git


In [ ]:
# Set working directory to the auto_labeler folder on Google Drive
%cd /content/drive/MyDrive/auto_labeler

/content/drive/MyDrive/auto_labeler


In [ ]:
# Verify directory contents
import os
print("Contents of working directory:")
for f in sorted(os.listdir(".")):
    print(f"  {f}")


Contents of working directory:
  __init__.py
  __pycache__
  checkpoints
  config.py
  dataset.py
  evaluate.py
  model.py
  predict.py
  predict_labels
  results
  test_data_8
  train.py
  train_colab.ipynb
  train_colab_v0.ipynb
  training_data


In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [ ]:
# Install any missing dependencies
!pip install pandas pyarrow matplotlib -q

In [ ]:
# Verify training data exists
import os
data_dir = "training_data"
files = [f for f in os.listdir(data_dir) if f.endswith('.parquet')]
print(f"Training parquets found: {len(files)}")
assert len(files) >= 85, f"Expected 85+ files, got {len(files)}. Check data_dir path."

Training parquets found: 85


In [ ]:
# Verify model builds correctly
from model import UNet1D, count_parameters

model = UNet1D(in_channels=1, num_classes=3, base_filters=32, depth=5, kernel_size=7)
print(f"Parameters: {count_parameters(model):,}")

x = torch.randn(2, 1, 4096).cuda()
model = model.cuda()
y = model(x)
print(f"Input: {x.shape} -> Output: {y.shape}")
del model, x, y
torch.cuda.empty_cache()

Parameters: 27,263,875
Input: torch.Size([2, 1, 4096]) -> Output: torch.Size([2, 3, 4096])


## Train

With A100 + batch_size=64, expect ~20 minutes for 80 epochs, else with cpu, expect ~ 40 hours.

In [ ]:
# Run training from parent directory so package imports work
%cd /content/drive/MyDrive

!python -m auto_labeler.train \
    --data_dir auto_labeler/training_data \
    --output_dir auto_labeler/checkpoints \
    --epochs 80 \
    --batch_size 64 \
    --lr 1e-3 \
    --base_filters 32 \
    --depth 5 \
    --kernel_size 7 \
    --patience 15 \
    --num_workers 2 \
    --seed 42

# Return to auto_labeler dir
%cd /content/drive/MyDrive/auto_labeler

## Evaluate on Test Studies

In [ ]:
# Evaluate on test studies (if they exist in test_data/)
%cd /content/drive/MyDrive

!python -m auto_labeler.evaluate \
    --data_dir auto_labeler/test_data_8 \
    --checkpoint auto_labeler/checkpoints/best_model.pt \
    --output_dir auto_labeler/results \
    --studies 33CFB812 819421BC 847A1E3F 8ECEADA6 CENT0008 DD2DFAF4 F427536B SUMM0127 \
    --plot

%cd /content/drive/MyDrive/auto_labeler

/content/drive/MyDrive
Device: cuda
Multichannel: False
Evaluating 8 studies...
  33CFB812: F1=0.6359, Acc=0.9025
  Saved: auto_labeler/results/plots/33CFB812_overlay.png
  819421BC: F1=0.7214, Acc=0.9051
  Saved: auto_labeler/results/plots/819421BC_overlay.png
  847A1E3F: F1=0.7041, Acc=0.8985
  Saved: auto_labeler/results/plots/847A1E3F_overlay.png
  8ECEADA6: F1=0.7973, Acc=0.8975
  Saved: auto_labeler/results/plots/8ECEADA6_overlay.png
  CENT0008: F1=0.7750, Acc=0.9005
  Saved: auto_labeler/results/plots/CENT0008_overlay.png
  DD2DFAF4: F1=0.5429, Acc=0.7859
  Saved: auto_labeler/results/plots/DD2DFAF4_overlay.png
  F427536B: F1=0.9272, Acc=0.9626
  Saved: auto_labeler/results/plots/F427536B_overlay.png
  SUMM0127: F1=0.6509, Acc=0.8742
  Saved: auto_labeler/results/plots/SUMM0127_overlay.png

SUMMARY (8 studies)
  Accuracy:   0.8909 ± 0.0493
  F1 macro:   0.7193 ± 0.1167
  F1 blood : 0.9548 ± 0.0164
  F1 clot  : 0.3947 ± 0.2538
  F1 wall  : 0.8085 ± 0.1174
Results saved: auto_labe

In [ ]:
# View a sample overlay plot
from IPython.display import Image
import glob

plots = sorted(glob.glob("results/plots/*.png"))
if plots:
    print(f"Found {len(plots)} plots")
    display(Image(plots[0], width=1000))
else:
    print("No plots generated (test studies may not be in training_data/)")

## Download Trained Model

In [ ]:
# Checkpoint is already on Google Drive (saved in-place during training)
print("Checkpoint location: /content/drive/MyDrive/auto_labeler/checkpoints/best_model.pt")
!ls -la checkpoints/best_model.pt

## Predict Labels on New Data

Place unlabeled parquet files in `auto_labeler/predict_labels/`. Each file should have columns `timeInMS` and `magRLoadAdjusted`. The model will add a `predicted_label` column (0=blood, 1=clot, 2=wall) and overwrite the file in-place.

In [ ]:
%cd /content/drive/MyDrive/auto_labeler
import sys
sys.path.insert(0, "/content/drive/MyDrive")

import os
print(f"CWD: {os.getcwd()}")
print(f"Checkpoint exists: {os.path.exists('checkpoints/best_model.pt')}")
print(f"predict_labels/ exists: {os.path.exists('predict_labels')}")
if os.path.exists('predict_labels'):
    print(f"  Contents: {os.listdir('predict_labels')}")

import pandas as pd
import numpy as np
from pathlib import Path
from auto_labeler.predict import load_model, predict_file
from auto_labeler.dataset import build_multichannel
import torch

checkpoint = "checkpoints/best_model.pt"
predict_dir = Path("predict_labels")
predict_dir.mkdir(exist_ok=True)

# Find all parquet files to label
parquets = sorted(predict_dir.glob("*.parquet"))
print(f"\nFound {len(parquets)} files to label")

if not parquets:
    print("No .parquet files found. Place files in /content/drive/MyDrive/auto_labeler/predict_labels/")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, multichannel = load_model(checkpoint, device)
    print(f"Model loaded (multichannel={multichannel})")

    for pf in parquets:
        print(f"\n  Processing: {pf.name}")
        df = pd.read_parquet(pf)
        resistance = df["magRLoadAdjusted"].values.astype(np.float32)

        # Build features matching training
        if multichannel:
            features = build_multichannel(resistance)
        else:
            mean, std = resistance.mean(), resistance.std() + 1e-8
            features = ((resistance - mean) / std)[np.newaxis, :]

        # Run prediction
        pred_labels = predict_file(model, features, device)

        # Add predicted_label column
        df["predicted_label"] = pred_labels

        # Save back
        df.to_parquet(pf, index=False)

        # Summary
        unique, counts = np.unique(pred_labels, return_counts=True)
        label_names = {0: "blood", 1: "clot", 2: "wall"}
        dist = ", ".join(f"{label_names[u]}: {c/len(pred_labels)*100:.1f}%" for u, c in zip(unique, counts))
        print(f"    Samples: {len(pred_labels):,} | {dist}")

    print(f"\nDone! {len(parquets)} files labeled and saved.")

/content/drive/MyDrive/auto_labeler
CWD: /content/drive/MyDrive/auto_labeler
Checkpoint exists: True
predict_labels/ exists: True
  Contents: ['33CFB812_labeled_segment.parquet']

Found 1 files to label
Model loaded (multichannel=False)

  Processing: 33CFB812_labeled_segment.parquet
    Samples: 351,627 | blood: 78.2%, clot: 2.1%, wall: 19.6%

Done! 1 files labeled and saved.
